# Caso 1 · Red + generación térmica + PV de tamaño fijo, sin BESS

Siguiente bloque incremental de la línea térmica. Se agrega al sistema del
cuaderno 11 una planta solar de tamaño fijo, **sin batería, sin exportación y
sin optimizar el tamaño**. La pregunta es qué cambia en la operación térmica,
en las compras de red y en los arranques cuando el PV cubre parte de la demanda.

```
RED  +  G1–G4  +  PV (con vertimiento)  ──►  DEMANDA
```

| capa | archivo | papel |
|---|---|---|
| Modelo | `termica_pypsa.py` | compromiso de unidades en PyPSA; `pv_disponible` agrega el PV sin costo variable |
| Caso 1 | `termica_pv.py` | entradas alineadas por fecha, validador independiente, indicadores y referencia económica |
| Pruebas | `termica_pv_pruebas.py` | ventanas cortas, regresión con PV = 0 y MILP independiente con PV |
| Cálculo | `termica_pv_anual.py` | año completo T0 y T1 con PV; reutiliza la referencia térmica anual validada |
| Lectura | este cuaderno | solo lee `resultados_nb12/` y `resultados_nb11/` |

El cuaderno 11 queda cerrado como referencia térmica. Este cuaderno **no resuelve
ningún modelo**: una guarda interrumpe la ejecución si alguna celda intentara
llamar a los solvers.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown

warnings.filterwarnings("ignore")
import costos
import termica as _T

SAL = Path("resultados_nb12")
NB11 = Path("resultados_nb11")
FIG = Path("figuras_nb12"); FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


# --- Guarda: este cuaderno solo lee resultados. Si algo intenta recalcular, para. ---
def _prohibido(nombre):
    def _f(*a, **k):
        raise RuntimeError("El cuaderno 12 no debe recalcular nada: se intentó llamar a {}. "
                           "Las fuentes son resultados_nb12/, que escribe termica_pv_anual.py.".format(nombre))
    return _f


for _nombre in ("cargar_entradas", "enumerar_termico", "resolver_compromiso_temporal",
                "resolver_eventos", "contraste_milp_compromiso"):
    setattr(_T, _nombre, _prohibido(_nombre))
sys.modules["termica_pypsa"] = None      # importar el modelo PyPSA falla a propósito


def es(x, dec=0):
    """Número con coma decimal y punto de miles, como pide la prosa del proyecto."""
    return f"{x:,.{dec}f}".replace(",", "§").replace(".", ",").replace("§", ".")


# Paleta validada con .build/validar_paleta.py sobre #fcfcfb: visión normal y tritan
# PASS; deutan/protan WARN en G3<->Red y PV con contraste 2,1:1, por eso las series
# llevan etiqueta directa y borde.
COLOR = {"G1+G2": "#2a78d6", "G3": "#a03060", "G4": "#0d5f4f", "Red": "#8b8a83", "PV": "#eda100"}
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#d8d8d4", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#d8d8d4", "font.size": 9.5})

RES = pd.read_csv(SAL/"pv_anual_resumen.csv", float_precision="round_trip")
DIF = pd.read_csv(SAL/"pv_anual_diferencias.csv", float_precision="round_trip")
PRU = pd.read_csv(SAL/"pv_pruebas.csv", float_precision="round_trip")
VER = pd.read_csv(SAL/"pv_verificaciones.csv")
HORA = pd.read_csv(SAL/"pv_arranques_por_hora.csv")
ECO = pd.read_csv(SAL/"pv_economia_referencia.csv", float_precision="round_trip")
assert (PRU.Estado == "OK").all() and (VER.Estado == "OK").all()
assert len(RES) == 4 and set(RES.Caso) == {"sin_PV", "con_PV"}
fila = lambda s, caso: RES[(RES.Senal == s) & (RES.Caso == caso)].iloc[0]
print("TRM {} COP/USD; {} pruebas cortas y {} controles anuales en OK.".format(
    es(costos.TRM_COP_USD, 2), len(PRU), len(VER)))

TRM 3.126,08 COP/USD; 13 pruebas cortas y 4 controles anuales en OK.


## 1 · Supuestos

**Se mantienen sin cambios** los de la referencia térmica anual validada
(cuaderno 11, §§26–29):

- demanda horaria limpia del cuaderno 02 y señales T0 y T1;
- G1–G4 con los HR de fabricante, mínimos técnicos del 30 % de la potencia
  disponible y gas a 16 USD/MMBtu;
- como máximo un arranque y una parada por unidad y día calendario;
- estado inicial libre, sin eventos en la primera hora ni parada final obligatoria;
- cargo de **100 USD por arranque, hipotético y no calibrado**, elegido porque ya
  existe una referencia anual validada con ese valor;
- 0 minutos sin generar por evento. Los descuentos de 10 y 20 minutos siguen
  disponibles en `resolver_pypsa`, pero aquí no se ejecutan.

**Se agrega:**

| | valor |
|---|---|
| Recurso solar | `solar_tmy.perfil_pv`: TMY NSRDB de la región **Oriente**, año 2025, transposición Perez, temperatura y pérdidas PVWatts, inversor |
| Tamaño | **PV_MAX_ALCANCE = 114,415019 MWp DC**, que da 95,35 MWac con DC/AC = 1,20 |
| Por qué ese tamaño | es el PV de la configuración representativa vigente (INFORME §0.3) y de K3; iguala la generación anual a la demanda anual (f_PV = 1). No se optimiza para este caso |
| Alineación | el perfil se une a la demanda **por fecha**, uno a uno; se verifica 2025, 8.760 h continuas y la coincidencia con el `PV_PU` del cuaderno 02 |
| Vertimiento | permitido: la potencia PV utilizada puede ser cualquier valor entre cero y la disponible |
| Exportación | **prohibida**: no hay componente de exportación; red y térmico no negativos y balance exacto |
| BESS | no hay |

El tamaño se definió para un sistema PV+BESS con exportación permitida. Sin
batería ni exportación, con una demanda media de 18 MW frente a un pico PV de
89,5 MW, se espera un vertimiento alto. Ese vertimiento es un **resultado** del
caso, no una recomendación de tamaño.

**Convención económica: PV propio**, la del cuaderno 02 y el factorial vigente.

- El PV no tiene costo variable: ni la energía usada ni la vertida se pagan,
  porque no hay PPA.
- El CAPEX sale de `costos.capex_pv` (curvas del Excel) y el OPEX anual es el 2 %
  del CAPEX. Se reportan **aparte** del costo operativo.
- La diferencia de costo operativo frente al caso térmico es un **ahorro
  operativo**, no una rentabilidad: faltan el CAPEX, el horizonte, la
  degradación y el descuento.

## 2 · Validación en ventanas cortas

`termica_pv_pruebas.py` ejecuta un caso sintético y tres ventanas reales, cada
una en T0 y T1:

- **caso sintético:** el mínimo técnico obliga a verter PV;
- **demanda baja con sol:** parada de planta del 10/04/2025, 48 h;
- **horas solares ordinarias:** 15/02/2025, 48 h;
- **cruce de medianoche al cierre del año:** 36 h.

En cada ventana:

1. **Regresión con PV = 0.** El camino con PV reproduce el costo de PyPSA sin PV
   y el de la programación dinámica, sin exigir despachos idénticos.
2. **PV de referencia.** Validación independiente —balance, PV utilizado +
   vertido = disponible, límites térmicos y de compromiso, ausencia de
   exportación, transiciones exactas, eventos diarios y costo reconstruido— y
   contraste contra un MILP de SciPy con PV, formulado aparte de PyPSA.
3. **Cota.** El costo con PV no supera el costo sin PV.

In [2]:
display(PRU)
print("error máximo frente al MILP con PV: {:.2e} USD".format(PRU.Error_MILP_USD.max()))
print("error máximo de la regresión PV = 0 frente a DP: {:.2e} USD; frente a PyPSA sin PV: {:.2e} USD".format(
    PRU.Error_DP_PV0_USD.max(), PRU.Error_PV0_vs_sin_PV_USD.max()))

,Prueba,Senal,PV_MWp,Horas,Costo_PyPSA_USD,Error_MILP_USD,Error_DP_PV0_USD,Error_PV0_vs_sin_PV_USD,PV_disponible_MWh,PV_vertido_MWh,Arranques,Brecha,Segundos,Estado
0,sintetica_vertimiento_por_minimo,-,NaN,3,1240.000000,0.000000e+00,NaN,NaN,10.000000,7.400000,0,0.000000e+00,1.212039,OK
1,demanda_baja_con_sol_PV0,T0,0.000000,48,31962.746290,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0,0.000000e+00,0.954694,OK
2,demanda_baja_con_sol_PV_referencia,T0,114.415019,48,26605.941632,7.275958e-12,NaN,NaN,1007.437646,981.335232,0,6.836779e-16,0.955001,OK
3,demanda_baja_con_sol_PV0,T1,0.000000,48,30216.072787,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0,0.000000e+00,1.063560,OK
4,demanda_baja_con_sol_PV_referencia,T1,114.415019,48,25636.181883,7.275958e-12,NaN,NaN,1007.437646,981.335232,0,0.000000e+00,0.952628,OK
5,horas_solares_PV0,T0,0.000000,48,131557.922729,NaN,2.910383e-11,1.018634e-09,0.000000,0.000000,2,0.000000e+00,1.008232,OK
6,horas_solares_PV_referencia,T0,114.415019,48,74502.889019,1.455192e-11,NaN,NaN,1066.082683,719.132840,5,0.000000e+00,1.182973,OK
7,horas_solares_PV0,T1,0.000000,48,131712.063381,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,2,0.000000e+00,1.119506,OK
8,horas_solares_PV_referencia,T1,114.415019,48,74545.329474,2.910383e-11,NaN,NaN,1066.082683,719.132840,6,0.000000e+00,1.099635,OK
9,cruce_medianoche_fin_de_anio_PV0,T0,0.000000,36,120698.165778,NaN,1.455192e-11,0.000000e+00,0.000000,0.000000,0,0.000000e+00,0.970762,OK


error máximo frente al MILP con PV: 2.91e-11 USD
error máximo de la regresión PV = 0 frente a DP: 2.91e-11 USD; frente a PyPSA sin PV: 1.02e-09 USD


## 3 · Resultados anuales

**Sin PV** se reutiliza la trayectoria PyPSA anual del cuaderno 11
(`resultados_nb11/pypsa_anual_{T0,T1}.npz`). Sus entradas y supuestos coinciden
con los de este caso, y `termica_pv_anual.py` lo comprueba antes de usarla: la
revalida con el mismo validador y la contrasta con la programación dinámica.
**Con PV** se resuelve el año completo en PyPSA. Un resultado solo se acepta si
HiGHS certifica el óptimo; una solución detenida por tiempo haría fallar el
script.

In [3]:
columnas = ["Senal", "Caso", "PV_MWp", "PV_disponible_MWh", "PV_utilizado_MWh", "PV_vertido_MWh",
            "Vertimiento_pct", "Energia_termica_MWh", "Gas_MMBtu", "Compras_red_MWh",
            "Arranques", "Arranques_G1_G2", "Arranques_G3", "Arranques_G4",
            "Costo_combustible_USD", "Costo_CVOM_USD", "Costo_red_USD", "Costo_arranques_USD",
            "Costo_operativo_total_USD", "Estado_solver", "Brecha", "Cota_dual_USD", "Segundos"]
display(RES[columnas].round(2))
display(DIF.round(2))
display(VER)

,Senal,Caso,PV_MWp,PV_disponible_MWh,PV_utilizado_MWh,PV_vertido_MWh,Vertimiento_pct,Energia_termica_MWh,Gas_MMBtu,Compras_red_MWh,Arranques,Arranques_G1_G2,Arranques_G3,Arranques_G4,Costo_combustible_USD,Costo_CVOM_USD,Costo_red_USD,Costo_arranques_USD,Costo_operativo_total_USD,Estado_solver,Brecha,Cota_dual_USD,Segundos
0,T0,sin_PV,0.00,0.00,0.00,0.00,0.00,158354.09,1560885.27,388.48,120,37,83,0,24974164.38,1469525.96,81686.16,12000.0,26537376.50,optimal (corrida validada del 13/09/2026),0.0,26537376.50,98.75
1,T0,con_PV,114.42,158742.57,62725.98,96016.58,60.49,94856.78,933874.93,1159.80,1018,723,293,2,14941998.89,880270.94,243852.85,101800.0,16167922.67,ok/optimal,0.0,16167922.67,109.62
2,T1,sin_PV,0.00,0.00,0.00,0.00,0.00,147226.56,1437905.99,11516.00,197,37,148,12,23006495.77,1366262.48,1969632.80,19700.0,26362091.05,optimal (corrida validada del 13/09/2026),0.0,26362091.05,98.41
3,T1,con_PV,114.42,158742.57,62717.61,96024.96,60.49,88322.42,862503.81,7702.54,975,723,232,20,13800060.98,819632.03,1330393.08,97500.0,16047586.09,ok/optimal,0.0,16047586.09,110.69


,Senal,Delta_PV_utilizado_MWh,Delta_PV_vertido_MWh,Delta_Energia_termica_MWh,Delta_Energia_G1_G2_MWh,Delta_Energia_G3_MWh,Delta_Energia_G4_MWh,Delta_Compras_red_MWh,Delta_Gas_MMBtu,Delta_Costo_combustible_USD,Delta_Costo_CVOM_USD,Delta_Costo_red_USD,Delta_Costo_arranques_USD,Delta_Costo_operativo_total_USD,Delta_Arranques,Delta_Arranques_G1_G2,Delta_Arranques_G3,Delta_Arranques_G4,Delta_Paradas,Delta_Horas_unidad_encendida_G1_G2,Delta_Horas_encendida_G3,Delta_Horas_encendida_G4,Ahorro_operativo_USD,Ahorro_operativo_pct
0,T0,62725.98,96016.58,-63497.31,-53092.39,-10469.41,64.50,771.32,-627010.34,-10032165.49,-589255.02,162166.68,89800.0,-10369453.83,898,686,210,2,898,-6529,-2976,23,10369453.83,39.07
1,T1,62717.61,96024.96,-58904.14,-53482.27,-5580.95,159.07,-3813.46,-575402.17,-9206434.79,-546630.45,-639239.71,77800.0,-10314504.96,778,686,84,8,778,-6625,-1664,67,10314504.96,39.13


,Senal,Caso,Control,Valor,Estado
0,T0,sin_PV,"Referencia reutilizada: supuestos, validación ...",3.725290e-09,OK
1,T0,con_PV,"Óptimo certificado, validación independiente y...",0.000000e+00,OK
2,T1,sin_PV,"Referencia reutilizada: supuestos, validación ...",0.000000e+00,OK
3,T1,con_PV,"Óptimo certificado, validación independiente y...",0.000000e+00,OK


In [4]:
# A. Comparación anual: costos, abastecimiento y arranques, T0 y T1, sin y con PV.
orden = [("T0", "sin_PV"), ("T0", "con_PV"), ("T1", "sin_PV"), ("T1", "con_PV")]
etiquetas = ["T0\nsin PV", "T0\ncon PV", "T1\nsin PV", "T1\ncon PV"]
x = np.array([0, 1, 2.6, 3.6])
F = [fila(s, c) for s, c in orden]
fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.8), layout="constrained")

ax = axs[0]
componentes = [("Térmico: combustible + CVOM", lambda f: f.Costo_combustible_USD + f.Costo_CVOM_USD, COLOR["G1+G2"]),
               ("Compras de red", lambda f: f.Costo_red_USD, COLOR["Red"]),
               ("Cargos de arranque", lambda f: f.Costo_arranques_USD, COLOR["G3"])]
base = np.zeros(4)
for nombre, valor, color in componentes:
    v = np.array([valor(f) for f in F])/1e6
    ax.bar(x, v, 0.8, bottom=base, color=color, label=nombre, edgecolor="white", linewidth=0.5)
    base += v
for xi, total in zip(x, base):
    ax.text(xi, total+0.3, es(total, 2), ha="center", fontsize=8.5)
ax.set_ylabel("Costo operativo anual [MUSD]")
ax.set_title("Costo operativo desagregado", loc="left", fontweight="bold")

ax = axs[1]
fuentes = [("G1+G2", "Energia_G1_G2_MWh"), ("G3", "Energia_G3_MWh"), ("G4", "Energia_G4_MWh"),
           ("Red", "Compras_red_MWh"), ("PV", "PV_utilizado_MWh")]
base = np.zeros(4)
for nombre, col in fuentes:
    v = np.array([f[col] for f in F])/1e3
    ax.bar(x, v, 0.8, bottom=base, color=COLOR[nombre], label=nombre if nombre != "PV" else "PV utilizado",
           edgecolor="#52514e" if nombre == "PV" else "white", linewidth=0.5)
    base += v
for xi, f, total in zip(x, F, base):
    if f.PV_disponible_MWh > 0:
        ax.text(xi, total+2, "vertido\n" + es(f.PV_vertido_MWh/1e3, 1) + " GWh", ha="center", fontsize=8)
ax.set_ylabel("Energía anual [GWh]")
ax.set_ylim(0, base.max()*1.18)
ax.set_title("Abastecimiento de la demanda", loc="left", fontweight="bold")

ax = axs[2]
bloques = [("G1+G2", "Arranques_G1_G2"), ("G3", "Arranques_G3"), ("G4", "Arranques_G4")]
base = np.zeros(4)
for nombre, col in bloques:
    v = np.array([f[col] for f in F], float)
    ax.bar(x, v, 0.8, bottom=base, color=COLOR[nombre], label=nombre, edgecolor="white", linewidth=0.5)
    base += v
for xi, total in zip(x, base):
    ax.text(xi, total+5, es(total), ha="center", fontsize=8.5)
ax.set_ylabel("Arranques por año")
ax.set_title("Arranques por bloque", loc="left", fontweight="bold")

for ax in axs:
    ax.set_xticks(x, etiquetas)
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=2)
fig.suptitle("Caso 1 · efecto de {} MWp de PV sin BESS · gas 16 USD/MMBtu, cargo hipotético de 100 USD/arranque".format(
    es(fila("T0", "con_PV").PV_MWp, 3)), fontweight="bold", x=0.01, ha="left")
fig.savefig(FIG/"12_1_comparacion_anual.png", dpi=160, bbox_inches="tight")
plt.show()

In [5]:
# B. Trayectorias bajo T1: los tres días con mayor cambio de arranques de G1+G2.
z0 = np.load(NB11/"pypsa_anual_T1.npz", allow_pickle=False)
z1 = np.load(SAL/"pv_anual_T1.npz", allow_pickle=False)
fechas = pd.DatetimeIndex(z1["Fecha"])
assert (pd.DatetimeIndex(z0["Fecha"]) == fechas).all()
carga = z1["potencia"].sum(axis=1) + z1["pv_usado"]
cambio = np.abs(z1["arranques"][:, :2].sum(1) - z0["arranques"][:, :2].sum(1))
inicio = int(np.argmax(np.convolve(cambio, np.ones(72), mode="valid")))
inicio -= inicio % 24
v = slice(inicio, inicio+72)
fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True, layout="constrained")
for ax, z, titulo in [(axs[0], z0, "T1 · sin PV"), (axs[1], z1, "T1 · con PV")]:
    p = z["potencia"][v]
    pv_usado = z["pv_usado"][v] if "pv_usado" in z.files else np.zeros(72)
    capas = [p[:, 0]+p[:, 1], p[:, 2], p[:, 3], p[:, 4], pv_usado]
    ax.stackplot(np.arange(72), capas, colors=[COLOR[k] for k in ("G1+G2", "G3", "G4", "Red", "PV")],
                 labels=["G1+G2", "G3", "G4", "Red", "PV utilizado"], step="post")
    if "pv_disponible" in z.files:
        ax.step(np.arange(72), z["pv_disponible"][v], where="post", color="#52514e", lw=1, ls=":",
                label="PV disponible")
    ax.step(np.arange(72), carga[v], where="post", color="#0b0b0b", lw=1.4, ls="--", label="Demanda")
    arr = z["arranques"][v][:, :2].sum(1)
    par = z["paradas"][v][:, :2].sum(1)
    # Eventos como marcadores sobre el área apilada, para que no tapen el despacho.
    ax.scatter(np.flatnonzero(arr)+0.5, np.full(int((arr > 0).sum()), 32.5), marker="^", s=45,
               color=COLOR["G1+G2"], edgecolor="#0b0b0b", linewidth=0.5, zorder=5, label="Arranque de G1 o G2")
    ax.scatter(np.flatnonzero(par)+0.5, np.full(int((par > 0).sum()), 32.5), marker="v", s=45,
               color="#fcfcfb", edgecolor="#0b0b0b", linewidth=0.8, zorder=5, label="Parada de G1 o G2")
    ax.set_ylim(0, 35)
    ax.set_ylabel("Potencia [MW]")
    ax.set_title(titulo, loc="left", fontweight="bold")
    ax.grid(axis="x", visible=False)
axs[1].set_xticks(np.arange(0, 73, 6), [(fechas[inicio] + pd.Timedelta(hours=int(h))).strftime("%d/%m %H:%M")
                                         for h in range(0, 73, 6)], rotation=30, ha="right")
fig.suptitle("Tres días bajo T1 con el mayor cambio de arranques de G1+G2 · el eje se recorta en 35 MW; "
             "el PV disponible supera ese valor al mediodía", fontweight="bold", x=0.01, ha="left", fontsize=10)
fig.legend(*axs[1].get_legend_handles_labels(), frameon=False, fontsize=8, ncol=5, loc="outside lower center")
fig.savefig(FIG/"12_2_dias_arranques_T1.png", dpi=160, bbox_inches="tight")
plt.show()

## 4 · Lectura de los resultados

In [6]:
texto = []
for s in ("T0", "T1"):
    a, b, d = fila(s, "sin_PV"), fila(s, "con_PV"), DIF[DIF.Senal == s].iloc[0]
    sentido = "suben" if d.Delta_Arranques > 0 else ("bajan" if d.Delta_Arranques < 0 else "no cambian")
    texto.append(
        f"- **{s}.** El PV dispone de {es(b.PV_disponible_MWh/1e3, 1)} GWh, utiliza {es(b.PV_utilizado_MWh/1e3, 1)} GWh "
        f"({es(b.Participacion_PV_demanda_pct, 1)} % de la demanda) y vierte {es(b.PV_vertido_MWh/1e3, 1)} GWh "
        f"({es(b.Vertimiento_pct, 1)} % de lo disponible). La energía térmica pasa de {es(a.Energia_termica_MWh/1e3, 1)} "
        f"a {es(b.Energia_termica_MWh/1e3, 1)} GWh, el gas de {es(a.Gas_MMBtu/1e3)} a {es(b.Gas_MMBtu/1e3)} mil MMBtu "
        f"y las compras de red de {es(a.Compras_red_MWh/1e3, 2)} a {es(b.Compras_red_MWh/1e3, 2)} GWh. "
        f"El costo operativo baja de {es(a.Costo_operativo_total_USD/1e6, 3)} a {es(b.Costo_operativo_total_USD/1e6, 3)} MUSD: "
        f"un **ahorro operativo** de {es(d.Ahorro_operativo_USD/1e6, 3)} MUSD ({es(d.Ahorro_operativo_pct, 1)} %). "
        f"Los arranques {sentido}: {es(a.Arranques)} → {es(b.Arranques)} "
        f"(G1+G2 {es(a.Arranques_G1_G2)} → {es(b.Arranques_G1_G2)}, G3 {es(a.Arranques_G3)} → {es(b.Arranques_G3)}, "
        f"G4 {es(a.Arranques_G4)} → {es(b.Arranques_G4)}); los cargos de arranque pasan de "
        f"{es(a.Costo_arranques_USD)} a {es(b.Costo_arranques_USD)} USD.")

# Explicación de los arranques desde las trayectorias: en qué horas ocurren.
h = HORA.set_index(["Senal", "Caso", "Hora"])
for s in ("T0", "T1"):
    con = h.loc[(s, "con_PV")]; sin = h.loc[(s, "sin_PV")]
    dia = con.loc[7:17]
    texto.append(
        f"- **{s}, cuándo ocurren.** Con PV, G1+G2 registran {int(con.Paradas_G1_G2.loc[5:12].sum())} paradas entre las "
        f"05:00 y las 12:59 y {int(con.Arranques_G1_G2.loc[13:20].sum())} arranques entre las 13:00 y las 20:59; sin PV "
        f"fueron {int(sin.Paradas_G1_G2.loc[5:12].sum())} y {int(sin.Arranques_G1_G2.loc[13:20].sum())}. "
        f"La hora con más arranques de G1+G2 con PV es las {int(con.Arranques_G1_G2.idxmax()):02d}:00.")
texto += ["",
    "**Cómo se explica el cambio en los arranques.** El límite permite un arranque y una parada por unidad "
    "y día. Una unidad base puede apagarse cuando el PV cubre la demanda y volver a encenderse al caer el sol; "
    "el óptimo lo hace cuando el combustible ahorrado en esas horas supera el cargo hipotético de 100 USD. "
    "Mantenerla encendida al mínimo obligaría, en cambio, a verter la energía solar que ese mínimo desplaza. "
    "El número de arranques depende del cargo supuesto. La figura B muestra la secuencia en los tres días "
    "con mayor cambio."]

# Vertimiento: parte explicada por exceso solar sobre la demanda y parte por generación
# térmica o compras mantenidas mientras había PV disponible.
for s in ("T0", "T1"):
    z = np.load(SAL/f"pv_anual_{s}.npz", allow_pickle=False)
    demanda = z["potencia"].sum(axis=1) + z["pv_usado"]
    vertido = z["pv_disponible"] - z["pv_usado"]
    por_exceso = np.minimum(vertido, np.maximum(z["pv_disponible"] - demanda, 0.)).sum()
    con_termico = vertido[z["potencia"][:, :4].sum(axis=1) > 1e-6].sum()
    total = vertido.sum()
    texto.append(
        f"- **{s}, vertimiento.** De {es(total/1e3, 1)} GWh vertidos, {es(por_exceso/1e3, 1)} GWh "
        f"({es(100*por_exceso/total, 1)} %) se deben a que el PV disponible supera la demanda de esa hora; "
        f"{es((total-por_exceso)/1e3, 2)} GWh restantes se vierten mientras otra fuente abastece parte de la "
        f"demanda. En horas con alguna unidad térmica encendida se vierten {es(con_termico/1e3, 2)} GWh.")
texto += ["",
    "El tamaño se tomó del caso PV+BESS con exportación, así que este vertimiento no dice nada sobre el "
    "tamaño adecuado para un sistema sin almacenamiento."]
display(Markdown("\n".join(texto)))

- **T0.** El PV dispone de 158,7 GWh, utiliza 62,7 GWh (39,5 % de la demanda) y vierte 96,0 GWh (60,5 % de lo disponible). La energía térmica pasa de 158,4 a 94,9 GWh, el gas de 1.561 a 934 mil MMBtu y las compras de red de 0,39 a 1,16 GWh. El costo operativo baja de 26,537 a 16,168 MUSD: un **ahorro operativo** de 10,369 MUSD (39,1 %). Los arranques suben: 120 → 1.018 (G1+G2 37 → 723, G3 83 → 293, G4 0 → 2); los cargos de arranque pasan de 12.000 a 101.800 USD.
- **T1.** El PV dispone de 158,7 GWh, utiliza 62,7 GWh (39,5 % de la demanda) y vierte 96,0 GWh (60,5 % de lo disponible). La energía térmica pasa de 147,2 a 88,3 GWh, el gas de 1.438 a 863 mil MMBtu y las compras de red de 11,52 a 7,70 GWh. El costo operativo baja de 26,362 a 16,048 MUSD: un **ahorro operativo** de 10,315 MUSD (39,1 %). Los arranques suben: 197 → 975 (G1+G2 37 → 723, G3 148 → 232, G4 12 → 20); los cargos de arranque pasan de 19.700 a 97.500 USD.
- **T0, cuándo ocurren.** Con PV, G1+G2 registran 712 paradas entre las 05:00 y las 12:59 y 713 arranques entre las 13:00 y las 20:59; sin PV fueron 17 y 14. La hora con más arranques de G1+G2 con PV es las 17:00.
- **T1, cuándo ocurren.** Con PV, G1+G2 registran 706 paradas entre las 05:00 y las 12:59 y 713 arranques entre las 13:00 y las 20:59; sin PV fueron 17 y 14. La hora con más arranques de G1+G2 con PV es las 17:00.

**Cómo se explica el cambio en los arranques.** El límite permite un arranque y una parada por unidad y día. Una unidad base puede apagarse cuando el PV cubre la demanda y volver a encenderse al caer el sol; el óptimo lo hace cuando el combustible ahorrado en esas horas supera el cargo hipotético de 100 USD. Mantenerla encendida al mínimo obligaría, en cambio, a verter la energía solar que ese mínimo desplaza. El número de arranques depende del cargo supuesto. La figura B muestra la secuencia en los tres días con mayor cambio.
- **T0, vertimiento.** De 96,0 GWh vertidos, 96,0 GWh (99,9 %) se deben a que el PV disponible supera la demanda de esa hora; 0,05 GWh restantes se vierten mientras otra fuente abastece parte de la demanda. En horas con alguna unidad térmica encendida se vierten 0,18 GWh.
- **T1, vertimiento.** De 96,0 GWh vertidos, 96,0 GWh (99,9 %) se deben a que el PV disponible supera la demanda de esa hora; 0,06 GWh restantes se vierten mientras otra fuente abastece parte de la demanda. En horas con alguna unidad térmica encendida se vierten 0,61 GWh.

El tamaño se tomó del caso PV+BESS con exportación, así que este vertimiento no dice nada sobre el tamaño adecuado para un sistema sin almacenamiento.

## 5 · Referencia económica del PV propio

El ahorro operativo se contrasta con los costos fijos del PV, sin mezclarlos
con el despacho. La tabla muestra el CAPEX de la curva vigente, el OPEX anual
(2 % del CAPEX) y el ahorro operativo del año 1 menos ese OPEX.

No es un VPN ni una rentabilidad:
- no incluye CAPEX, financiamiento, degradación ni horizonte;
- la configuración PV se definió para un sistema con BESS y exportación.

In [7]:
display(ECO)
for _, e in ECO.iterrows():
    print("{}: CAPEX PV {} MUSD; OPEX PV {} MUSD/año; ahorro operativo {} MUSD/año; ahorro − OPEX {} MUSD/año".format(
        e.Senal, es(e.CAPEX_PV_USD/1e6, 2), es(e.OPEX_PV_anual_USD/1e6, 3),
        es(e.Ahorro_operativo_anual_USD/1e6, 3), es(e.Ahorro_operativo_menos_OPEX_PV_USD/1e6, 3)))

,Senal,PV_MWp,PV_MWac,CAPEX_PV_USD,OPEX_PV_anual_USD,OPEX_fraccion_CAPEX,TRM_COP_USD,Fuente,Ahorro_operativo_anual_USD,Ahorro_operativo_menos_OPEX_PV_USD,Nota
0,T0,114.415019,95.345849,7.697881e+07,1.539576e+06,0.02,3126.08,costos.capex_pv (CurvasCapexPVBESS.xlsx) y cos...,1.036945e+07,8.829878e+06,"Año 1 sin degradación; no incluye CAPEX, finan..."
1,T1,114.415019,95.345849,7.697881e+07,1.539576e+06,0.02,3126.08,costos.capex_pv (CurvasCapexPVBESS.xlsx) y cos...,1.031450e+07,8.774929e+06,"Año 1 sin degradación; no incluye CAPEX, finan..."


T0: CAPEX PV 76,98 MUSD; OPEX PV 1,540 MUSD/año; ahorro operativo 10,369 MUSD/año; ahorro − OPEX 8,830 MUSD/año
T1: CAPEX PV 76,98 MUSD; OPEX PV 1,540 MUSD/año; ahorro operativo 10,315 MUSD/año; ahorro − OPEX 8,775 MUSD/año


## 6 · Limitaciones y decisiones pendientes

- **Tamaño PV no optimizado.** Se usó el PV de la configuración representativa
  PV+BESS. Sin BESS ni exportación una parte grande se vierte; un tamaño propio
  para este caso requiere una decisión de alcance antes de barrer tamaños.
- **Cargo de arranque hipotético.** El resultado sobre arranques depende de los
  100 USD supuestos; no hay costo calibrado.
- **Previsión perfecta.** Es una referencia retrospectiva con la serie anual
  conocida, no una política de operación implementable.
- **Sin rampas ni permanencias adicionales.** Tampoco se modela consumo de
  arranque ni la curva a carga parcial.
- **Referencia económica sin VPN.** No se evalúan degradación PV, horizonte ni
  financiamiento en este bloque.
- **Siguiente bloque:** agregar el BESS y decidir si se permite exportación.

## 7 · Diagnóstico exploratorio de horas equivalentes de operación (EOH)

¿Compensa la reducción de horas térmicas los arranques adicionales, al agregar PV
o al imponer un límite anual de arranques? `termica_eoh.py` lo explora **solo
desde las trayectorias ya guardadas**: no optimiza ni cambia ningún resultado.
Escribe `resultados_nb12/eoh_*.csv`, que esta sección lee.

**Casos comparables.** Gas 16 USD/MMBtu, 100 USD/arranque (hipotético), 0 minutos
por evento, estado inicial libre, T0 y T1:
- red + térmico, sin PV;
- red + térmico + PV fijo;
- PV con límites anuales de 180, 60 y 0 arranques por unidad.

El límite 365 **es** el caso PV reutilizado: el máximo diario ya lo implica, así
que no es un experimento aparte.

**Reconstrucción por unidad.**
- H: horas encendida, suma del compromiso por el paso horario.
- N: arranques, transiciones de apagada a encendida. Estar encendida en la
  primera hora no cuenta, y no se inventa una parada al final.
- Energía generada.

Los conteos se contrastan con los CSV originales. G1 y G2 se reportan por
trazabilidad; el bloque G1+G2 es el principal, porque las gemelas admiten óptimos
alternativos. Los arranques del bloque son la suma de los individuales.

**Aproximación exploratoria:** `EOH(f) = H + f · N`, con f = 0, 10, 20 y 50
horas equivalentes por arranque. Frente a una referencia:

```
ΔEOH(f) = ΔH + f · ΔN        f* = −ΔH / ΔN   (si ΔN ≠ 0)
```

- Si **f* > 0**, a un lado del umbral domina el cambio de horas y al otro el de arranques.
- Si **f* < 0**, no hay cruce para f ≥ 0: el signo no cambia.
- Si **ΔN = 0**, no hay umbral.

El umbral de un bloque se calcula con ΔH y ΔN agregados, nunca promediando los
individuales, y supone un mismo f para sus unidades.

**Qué no es.**
- Los valores de f son **ilustrativos**: no provienen de fabricantes ni forman un
  rango certificado o probable. Usar el mismo f en todas las unidades simplifica.
- EOH **no es el contador real de mantenimiento**. No incorpora carga parcial,
  temperatura, tipo de arranque, disparos ni historial previo.
- Mide la acumulación en el año modelado, no la vida remanente ni las fechas de
  overhaul.
- Sumar unidades da un indicador agregado del escenario, no la vida de un equipo.
- No se convierte a dólares y no cambia el cargo de 100 USD.
- **T0 con límite 60** es una solución factible limitada por tiempo. Sus métricas
  son las de la trayectoria guardada; la brecha de costo (0,118 %) no acota horas,
  arranques ni EOH.

In [8]:
CA = pd.read_csv(SAL/"eoh_casos.csv", float_precision="round_trip")
UN = pd.read_csv(SAL/"eoh_unidades.csv", float_precision="round_trip")
EQ = pd.read_csv(SAL/"eoh_equilibrio.csv", float_precision="round_trip")
PE = pd.read_csv(SAL/"eoh_pruebas.csv")
assert (PE.Estado == "OK").all() and len(CA) == 10
print("{} pruebas del diagnóstico EOH en OK ({} sintéticas, {} sobre resultados).".format(
    len(PE), int((PE.Tipo == "sintética").sum()), int((PE.Tipo == "resultados").sum())))
display(CA[["Senal", "Caso", "Limite_anual_por_unidad", "Estado_solucion", "Brecha_costo",
            "Unidades_encendidas_hora_0", "Nota"]])
cols_eoh = ["EOH_f0_h", "EOH_f10_h", "EOH_f20_h", "EOH_f50_h"]
display(UN[UN.Unidad.isin(["G1", "G2", "G1+G2", "G3", "G4", "Total"])]
        [["Senal", "Caso", "Estado_solucion", "Unidad", "Horas_encendida_h", "Arranques", "Energia_MWh"] + cols_eoh]
        .round(0))
display(EQ[["Senal", "Comparacion", "Unidad", "Delta_H_h", "Delta_N", "Delta_EOH_f0_h", "Delta_EOH_f10_h",
            "Delta_EOH_f20_h", "Delta_EOH_f50_h", "f_equilibrio_h_por_arranque", "Tipo_umbral", "Estado_caso"]]
        .round(2))

37 pruebas del diagnóstico EOH en OK (21 sintéticas, 16 sobre resultados).


,Senal,Caso,Limite_anual_por_unidad,Estado_solucion,Brecha_costo,Unidades_encendidas_hora_0,Nota
0,T0,sin_PV,NaN,PRECISION_SOLICITADA,0.000000e+00,3,"trayectoria PyPSA anual del cuaderno 11, 0 min..."
1,T0,con_PV,365.0,PRECISION_SOLICITADA,0.000000e+00,3,caso PV sin límite anual adicional; el límite ...
2,T0,limite_180,180.0,PRECISION_SOLICITADA,5.625471e-15,3,brecha relativa ≤ 1e−6 solicitada
3,T0,limite_60,60.0,FACTIBLE_LIMITE_TIEMPO,1.602090e-03,3,Métricas de la trayectoria factible guardada. ...
4,T0,limite_0,0.0,PRECISION_SOLICITADA,2.302441e-07,3,brecha relativa ≤ 1e−6 solicitada
5,T1,sin_PV,NaN,PRECISION_SOLICITADA,0.000000e+00,3,"trayectoria PyPSA anual del cuaderno 11, 0 min..."
6,T1,con_PV,365.0,PRECISION_SOLICITADA,0.000000e+00,3,caso PV sin límite anual adicional; el límite ...
7,T1,limite_180,180.0,PRECISION_SOLICITADA,0.000000e+00,3,brecha relativa ≤ 1e−6 solicitada
8,T1,limite_60,60.0,PRECISION_SOLICITADA,9.209279e-07,3,brecha relativa ≤ 1e−6 solicitada
9,T1,limite_0,0.0,PRECISION_SOLICITADA,0.000000e+00,3,brecha relativa ≤ 1e−6 solicitada


,Senal,Caso,Estado_solucion,Unidad,Horas_encendida_h,Arranques,Energia_MWh,EOH_f0_h,EOH_f10_h,EOH_f20_h,EOH_f50_h
0,T0,sin_PV,PRECISION_SOLICITADA,G1,8649.0,20,67211.0,8649.0,8849.0,9049.0,9649.0
1,T0,sin_PV,PRECISION_SOLICITADA,G2,8660.0,17,67250.0,8660.0,8830.0,9000.0,9510.0
2,T0,sin_PV,PRECISION_SOLICITADA,G3,6508.0,83,23893.0,6508.0,7338.0,8168.0,10658.0
3,T0,sin_PV,PRECISION_SOLICITADA,G4,0.0,0,-0.0,0.0,0.0,0.0,0.0
4,T0,sin_PV,PRECISION_SOLICITADA,G1+G2,17309.0,37,134461.0,17309.0,17679.0,18049.0,19159.0
5,T0,sin_PV,PRECISION_SOLICITADA,Total,23817.0,120,158354.0,23817.0,25017.0,26217.0,29817.0
6,T0,con_PV,PRECISION_SOLICITADA,G1,5385.0,361,40666.0,5385.0,8995.0,12605.0,23435.0
7,T0,con_PV,PRECISION_SOLICITADA,G2,5395.0,362,40703.0,5395.0,9015.0,12635.0,23495.0
8,T0,con_PV,PRECISION_SOLICITADA,G3,3532.0,293,13423.0,3532.0,6462.0,9392.0,18182.0
9,T0,con_PV,PRECISION_SOLICITADA,G4,23.0,2,64.0,23.0,43.0,63.0,123.0


,Senal,Comparacion,Unidad,Delta_H_h,Delta_N,Delta_EOH_f0_h,Delta_EOH_f10_h,Delta_EOH_f20_h,Delta_EOH_f50_h,f_equilibrio_h_por_arranque,Tipo_umbral,Estado_caso
0,T0,PV_vs_sin_PV,G1,-3264.0,341,-3264.0,146.0,3556.0,13786.0,9.57,cruce,PRECISION_SOLICITADA
1,T0,PV_vs_sin_PV,G2,-3265.0,345,-3265.0,185.0,3635.0,13985.0,9.46,cruce,PRECISION_SOLICITADA
2,T0,PV_vs_sin_PV,G3,-2976.0,210,-2976.0,-876.0,1224.0,7524.0,14.17,cruce,PRECISION_SOLICITADA
3,T0,PV_vs_sin_PV,G4,23.0,2,23.0,43.0,63.0,123.0,-11.50,sin_cruce,PRECISION_SOLICITADA
4,T0,PV_vs_sin_PV,G1+G2,-6529.0,686,-6529.0,331.0,7191.0,27771.0,9.52,cruce,PRECISION_SOLICITADA
5,T0,PV_vs_sin_PV,Total,-9482.0,898,-9482.0,-502.0,8478.0,35418.0,10.56,cruce,PRECISION_SOLICITADA
6,T0,limite_180_vs_PV,G1,1025.0,-181,1025.0,-785.0,-2595.0,-8025.0,5.66,cruce,PRECISION_SOLICITADA
7,T0,limite_180_vs_PV,G2,709.0,-182,709.0,-1111.0,-2931.0,-8391.0,3.90,cruce,PRECISION_SOLICITADA
8,T0,limite_180_vs_PV,G3,-1139.0,-113,-1139.0,-2269.0,-3399.0,-6789.0,-10.08,sin_cruce,PRECISION_SOLICITADA
9,T0,limite_180_vs_PV,G4,1408.0,118,1408.0,2588.0,3768.0,7308.0,-11.93,sin_cruce,PRECISION_SOLICITADA


In [9]:
# C. ΔEOH del bloque G1+G2 frente a f, T0 y T1 en paneles separados.
# Paleta validada sobre #fcfcfb (normal, deutan y tritan PASS; protan 60<->0 ΔE 7,8 y
# contraste 2,7:1 del aqua): cada curva lleva etiqueta directa y la leyenda repite f*.
SERIES_EOH = [("PV_vs_sin_PV", "PV frente a sin PV", "PV", "#2a78d6"),
              ("limite_180_vs_PV", "Límite 180 frente a PV", "180", "#eb6834"),
              ("limite_60_vs_PV", "Límite 60 frente a PV", "60", "#a03060"),
              ("limite_0_vs_PV", "Límite 0 frente a PV", "0", "#1baf7a")]


def umbral_txt(q):
    if q.Tipo_umbral == "cruce":
        return "f* = {} h/arranque".format(es(q.f_equilibrio_h_por_arranque, 1))
    if q.Tipo_umbral == "sin_cruce":
        return "sin cruce para f ≥ 0"
    if q.Tipo_umbral == "umbral_cero":
        return "umbral en f = 0"
    return "sin umbral (ΔN = 0)"


fgrid = np.linspace(0, 50, 101)
FACT = [0, 10, 20, 50]
fig, axs = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True, layout="constrained")
for ax, s in zip(axs, ("T0", "T1")):
    ax.axhline(0, color="#52514e", lw=1)
    for comp, nombre, corto, color in SERIES_EOH:
        q = EQ[(EQ.Senal == s) & (EQ.Comparacion == comp) & (EQ.Unidad == "G1+G2")].iloc[0]
        tiempo = "FACTIBLE_LIMITE_TIEMPO" in (q.Estado_caso, q.Estado_referencia)
        y = (q.Delta_H_h + fgrid*q.Delta_N)/1e3
        etiqueta = "{} · {}{}".format(nombre, umbral_txt(q), " · factible por tiempo" if tiempo else "")
        ax.plot(fgrid, y, color=color, lw=2, ls="--" if tiempo else "-", label=etiqueta)
        ax.scatter(FACT, [(q.Delta_H_h + f*q.Delta_N)/1e3 for f in FACT], color=color, s=16, zorder=4)
        ax.text(50.8, y[-1], corto, color=color, va="center", fontsize=9, fontweight="bold")
        if q.Tipo_umbral == "cruce" and 0 <= q.f_equilibrio_h_por_arranque <= 50:
            ax.scatter([q.f_equilibrio_h_por_arranque], [0], s=48, facecolor="#fcfcfb", edgecolor=color,
                       linewidth=1.6, zorder=5)
    ax.set_xticks(FACT)
    ax.set_xlim(0, 50)
    ax.set_xlabel("f: horas equivalentes por arranque (ilustrativo)")
    ax.set_title(s, loc="left", fontweight="bold")
    ax.text(0.5, 0.97, "sobre cero: más EOH que la referencia", transform=ax.transAxes, ha="center",
            va="top", fontsize=8, color="#52514e")
    ax.text(0.5, 0.03, "bajo cero: menos EOH que la referencia", transform=ax.transAxes, ha="center",
            va="bottom", fontsize=8, color="#52514e")
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=1)
axs[0].set_ylabel("ΔEOH del bloque G1+G2 [miles de h equivalentes]")
fig.suptitle("Diagnóstico exploratorio: ΔEOH = ΔH + f·ΔN del bloque G1+G2 · círculo abierto: umbral f* · "
             "referencia sin PV para la curva PV y caso PV para los límites",
             fontweight="bold", x=0.01, ha="left", fontsize=10)
fig.savefig(FIG/"12_4_eoh_equilibrio_G1G2.png", dpi=160, bbox_inches="tight")
plt.show()

In [10]:
def mayor_menor(x):
    return "menor" if x < 0 else "mayor"


def lectura(q):
    if q.Tipo_umbral == "cruce":
        return ("con f < {} h/arranque la EOH es {} que la de la referencia; por encima de ese umbral es {}".format(
            es(q.f_equilibrio_h_por_arranque, 1), mayor_menor(q.Delta_H_h), mayor_menor(q.Delta_N)))
    if q.Tipo_umbral == "sin_cruce":
        return "la EOH es {} que la de la referencia para todo f ≥ 0".format(mayor_menor(q.Delta_N))
    if q.Tipo_umbral == "umbral_cero":
        return "coincide en f = 0 y es {} para f > 0".format(mayor_menor(q.Delta_N))
    if q.Tipo_umbral == "sin_cambio":
        return "no cambia"
    return "es {} para todo f; no hay umbral".format(mayor_menor(q.Delta_H_h))


eq = lambda s, c, u: EQ[(EQ.Senal == s) & (EQ.Comparacion == c) & (EQ.Unidad == u)].iloc[0]
LIM = pd.read_csv(SAL/"limites_arranques_resumen.csv", float_precision="round_trip")
texto = []
for s in ("T0", "T1"):
    partes = []
    for u in ("G1+G2", "G3", "G4", "Total"):
        q = eq(s, "PV_vs_sin_PV", u)
        partes.append("{}: ΔH = {} h, ΔN = {}; {}".format(u, es(q.Delta_H_h), es(q.Delta_N), lectura(q)))
    texto.append("- **{}, PV frente a sin PV.** ".format(s) + ". ".join(partes) + ".")
    for L in (180, 60, 0):
        q, t = eq(s, f"limite_{L}_vs_PV", "G1+G2"), eq(s, f"limite_{L}_vs_PV", "Total")
        red = LIM[(LIM.Senal == s) & (LIM.Limite_anual_por_unidad == L)].iloc[0]
        base = LIM[(LIM.Senal == s) & (LIM.Limite_anual_por_unidad == 365)].iloc[0]
        nota = " *(trayectoria factible por tiempo)*" if q.Estado_caso == "FACTIBLE_LIMITE_TIEMPO" else ""
        texto.append(
            "- **{}, límite {} frente a PV{}.** G1+G2: ΔH = {} h, ΔN = {}; {}. Total: {}. "
            "Energía térmica {} → {} GWh; compras de red {} → {} GWh.".format(
                s, L, nota, es(q.Delta_H_h), es(q.Delta_N), lectura(q), lectura(t),
                es(base.Energia_termica_MWh/1e3, 1), es(red.Energia_termica_MWh/1e3, 1),
                es(base.Compras_red_MWh/1e3, 1), es(red.Compras_red_MWh/1e3, 1)))
texto += ["",
    "**Cómo leerlo.** Con PV, G1+G2 operan unas 6.500 horas-unidad menos y arrancan unas 685 veces más. "
    "El umbral del bloque, cerca de 9,5 h/arranque en T0 y 9,7 en T1, separa dos lecturas: por debajo, "
    "el PV reduce la acumulación EOH; por encima, la aumenta. Con los límites anuales el signo depende de "
    "la señal. En T0, los límites 180 y 60 alargan la operación de G1+G2 a cambio de menos arranques, "
    "y solo reducen la EOH si cada arranque vale más que el umbral. En T1, los tres límites reducen horas "
    "y arranques de G1+G2 a la vez, sin cruce, porque las compras de red sustituyen generación térmica. "
    "Con límite 0 casi no queda generación térmica: las horas restantes vienen del estado inicial libre, "
    "con unidades encendidas desde la primera hora hasta su única parada.",
    "",
    "Una EOH menor no es por sí misma una mejora: los límites suben el costo operativo (tabla de límites) "
    "y la EOH no mide el contador real de mantenimiento. Estos umbrales son un diagnóstico exploratorio "
    "con f ilustrativo, no un criterio de decisión."]
display(Markdown("\n".join(texto)))

- **T0, PV frente a sin PV.** G1+G2: ΔH = -6.529 h, ΔN = 686; con f < 9,5 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor. G3: ΔH = -2.976 h, ΔN = 210; con f < 14,2 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor. G4: ΔH = 23 h, ΔN = 2; la EOH es mayor que la de la referencia para todo f ≥ 0. Total: ΔH = -9.482 h, ΔN = 898; con f < 10,6 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor.
- **T0, límite 180 frente a PV.** G1+G2: ΔH = 1.734 h, ΔN = -363; con f < 4,8 h/arranque la EOH es mayor que la de la referencia; por encima de ese umbral es menor. Total: con f < 5,6 h/arranque la EOH es mayor que la de la referencia; por encima de ese umbral es menor. Energía térmica 94,9 → 98,4 GWh; compras de red 1,2 → 3,1 GWh.
- **T0, límite 60 frente a PV *(trayectoria factible por tiempo)*.** G1+G2: ΔH = 5.018 h, ΔN = -603; con f < 8,3 h/arranque la EOH es mayor que la de la referencia; por encima de ese umbral es menor. Total: con f < 4,0 h/arranque la EOH es mayor que la de la referencia; por encima de ese umbral es menor. Energía térmica 94,9 → 98,4 GWh; compras de red 1,2 → 9,3 GWh.
- **T0, límite 0 frente a PV.** G1+G2: ΔH = -9.194 h, ΔN = -723; la EOH es menor que la de la referencia para todo f ≥ 0. Total: la EOH es menor que la de la referencia para todo f ≥ 0. Energía térmica 94,9 → 8,9 GWh; compras de red 1,2 → 88,5 GWh.
- **T1, PV frente a sin PV.** G1+G2: ΔH = -6.625 h, ΔN = 686; con f < 9,7 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor. G3: ΔH = -1.664 h, ΔN = 84; con f < 19,8 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor. G4: ΔH = 67 h, ΔN = 8; la EOH es mayor que la de la referencia para todo f ≥ 0. Total: ΔH = -8.222 h, ΔN = 778; con f < 10,6 h/arranque la EOH es menor que la de la referencia; por encima de ese umbral es mayor.
- **T1, límite 180 frente a PV.** G1+G2: ΔH = -3.851 h, ΔN = -363; la EOH es menor que la de la referencia para todo f ≥ 0. Total: la EOH es menor que la de la referencia para todo f ≥ 0. Energía térmica 88,3 → 58,3 GWh; compras de red 7,7 → 38,7 GWh.
- **T1, límite 60 frente a PV.** G1+G2: ΔH = -1.549 h, ΔN = -603; la EOH es menor que la de la referencia para todo f ≥ 0. Total: la EOH es menor que la de la referencia para todo f ≥ 0. Energía térmica 88,3 → 60,0 GWh; compras de red 7,7 → 42,1 GWh.
- **T1, límite 0 frente a PV.** G1+G2: ΔH = -9.100 h, ΔN = -723; la EOH es menor que la de la referencia para todo f ≥ 0. Total: la EOH es menor que la de la referencia para todo f ≥ 0. Energía térmica 88,3 → 10,7 GWh; compras de red 7,7 → 87,2 GWh.

**Cómo leerlo.** Con PV, G1+G2 operan unas 6.500 horas-unidad menos y arrancan unas 685 veces más. El umbral del bloque, cerca de 9,5 h/arranque en T0 y 9,7 en T1, separa dos lecturas: por debajo, el PV reduce la acumulación EOH; por encima, la aumenta. Con los límites anuales el signo depende de la señal. En T0, los límites 180 y 60 alargan la operación de G1+G2 a cambio de menos arranques, y solo reducen la EOH si cada arranque vale más que el umbral. En T1, los tres límites reducen horas y arranques de G1+G2 a la vez, sin cruce, porque las compras de red sustituyen generación térmica. Con límite 0 casi no queda generación térmica: las horas restantes vienen del estado inicial libre, con unidades encendidas desde la primera hora hasta su única parada.

Una EOH menor no es por sí misma una mejora: los límites suben el costo operativo (tabla de límites) y la EOH no mide el contador real de mantenimiento. Estos umbrales son un diagnóstico exploratorio con f ilustrativo, no un criterio de decisión.